In [0]:
"""Health check of the compute / teste rápido de saúde do compute."""

# trivial query to confirm the session responds / consulta trivial para confirmar que a sessão responde
display(spark.range(5))

id
0
1
2
3
4


In [0]:
"""Create the raw volume in bronze / cria o volume bruto no bronze."""

# create the volume if it does not exist / cria o volume se não existir
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files")

DataFrame[]

In [0]:
"""Install the Kaggle client / instala o cliente do Kaggle."""

!pip install kaggle --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""List dataset files without downloading / lista os arquivos do dataset sem baixar."""

import os
from getpass import getpass
from kaggle.api.kaggle_api_extended import KaggleApi

# credentials typed at runtime, never saved in the notebook / credenciais digitadas na hora, nunca salvas no notebook
os.environ["KAGGLE_USERNAME"] = input("Kaggle username: ")
os.environ["KAGGLE_KEY"] = getpass("Kaggle key: ")

Kaggle username:  gallo33henrique

Kaggle key:  [REDACTED]

In [0]:
# authenticate / autentica
api = KaggleApi()
api.authenticate()

# dataset slug / identificador do dataset
dataset = "karthikgangula/credit-card-fraud-mega-dataset"

# files and sizes in GB / arquivos e tamanhos em GB
for f in api.dataset_list_files(dataset).files:
    print(f.name, round(f.total_bytes / 1024**3, 2), "GB")

credit_card_fraud.csv 10.42 GB
customers.csv 0.0 GB


In [0]:
"""List the exact file names from Kaggle / lista os nomes exatos dos arquivos do Kaggle."""

# dataset slug / identificador do dataset
dataset = "karthikgangula/credit-card-fraud-mega-dataset"

# files with exact names, using repr to reveal spaces / arquivos com nomes exatos, usando repr para mostrar espaços
files = api.dataset_list_files(dataset).files
for f in files:
    print(repr(f.name), round(f.total_bytes / 1024**3, 2), "GB")

'credit_card_fraud.csv' 10.42 GB
'customers.csv' 0.0 GB


In [0]:
"""Download the small customers file / baixa o arquivo pequeno de clientes."""

# volume path / caminho do volume
volume_path = r"/Volumes/workspace/bronze/raw_files"

# exact file name from the kaggle listing / nome exato do arquivo conforme a listagem do kaggle
api.dataset_download_file(dataset, "customers.csv", path=volume_path, force=True)

# list what is in the volume / lista o que está no volume
for f in dbutils.fs.ls(volume_path):
    print(f.name, round(f.size / 1024**3, 3), "GB")

Dataset URL: https://www.kaggle.com/datasets/karthikgangula/credit-card-fraud-mega-dataset
customers.csv.zip 0.001 GB


In [0]:
"""Inspect the customers file / inspeciona o arquivo de clientes."""

import zipfile

# unzip if the download came compressed / descompacta se o download veio compactado
for f in dbutils.fs.ls(volume_path):
    if f.name.endswith(".zip"):
        with zipfile.ZipFile(rf"{volume_path}/{f.name}") as z:
            z.extractall(volume_path)
        print("extracted:", f.name)

# read and show schema and rows / lê e mostra schema e linhas
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rf"{volume_path}/customers.csv")
)
df_customers.printSchema()
display(df_customers.limit(10))
print(df_customers.count())

extracted: customers.csv.zip
root
 |-- ssn|cc_num|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|acct_num|profile: string (nullable = true)



ssn|cc_num|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|acct_num|profile
115-04-4507|4218196001337|Jonathan|Johnson|M|863 Lawrence Valleys|Freeport|ME|04032|43.857|-70.1031|7879|Accounting technician|1973-10-03|888022315787|adults_50up_male_urban.json
715-55-5575|4351161559407816183|Edward|Fuller|M|310 Kendra Common Apt. 164|Chicago|IL|60638|41.7814|-87.7705|2680484|Professor Emeritus|1981-06-07|917558277935|adults_2550_male_urban.json
167-48-5821|4192832764832|Melinda|Cameron|F|05641 Robin Port|Reno|NV|89509|39.498|-119.8239|276896|International aid/development worker|1994-05-30|718172762479|adults_2550_female_urban.json
406-83-7518|4238849696532874|Brandon|Williams|M|26916 Carlson Mountain|Lady Lake|FL|32162|28.9471|-81.9971|74047|Seismic interpreter|1971-12-26|947268892251|adults_50up_male_urban.json
697-93-1877|4514627048281480|Lee|Hernandez|M|809 Burns Creek|Hermosa Beach|CA|90254|33.8643|-118.3955|19506|Medical laboratory scientific officer|1966-05-22|888335239225|adults_50up_male_urban.json
246-08-3947|3517182278248964|Tracey|Hickman|F|465 Lam Mission|Los Angeles|CA|90011|34.0079|-118.2582|2383912|Community arts worker|1972-06-18|245877882397|adults_50up_female_urban.json
347-03-9639|213193010310510|Jose|Ramirez|M|82997 Kristen Valley Suite 311|Paris|TX|75462|33.6805|-95.4905|34942|Plant breeder/geneticist|1984-03-06|465378164447|adults_2550_male_urban.json
479-94-0888|4065133387262473|David|Leblanc|M|108 Tanya Terrace Apt. 267|Lancaster|PA|17601|40.0766|-76.3107|164596|Quarry manager|1981-12-27|950105810049|adults_2550_male_urban.json
061-22-6210|630447468723|Colton|Robinson|M|805 Brendan Neck|Lake|MI|48632|43.8575|-85.0219|5125|Retail buyer|2001-08-23|940251608167|young_adults_male_urban.json
162-08-8321|2222913619399092|Angela|Sandoval|F|3534 Anderson Rue Suite 751|Brownstown|IL|62418|38.9891|-88.9494|2133|Acupuncturist|1971-01-26|684868121383|adults_50up_female_rural.json


20000


In [0]:
"""Download the large transactions file / baixa o arquivo grande de transações."""

# exact file name from the kaggle listing / nome exato do arquivo conforme a listagem do kaggle
api.dataset_download_file(dataset, "credit_card_fraud.csv", path=volume_path, force=True)

# unzip if the download came compressed / descompacta se o download veio compactado
for f in dbutils.fs.ls(volume_path):
    if f.name.endswith(".zip"):
        with zipfile.ZipFile(rf"{volume_path}/{f.name}") as z:
            z.extractall(volume_path)
        print("extracted:", f.name)

# list what is in the volume / lista o que está no volume
for f in dbutils.fs.ls(volume_path):
    print(f.name, round(f.size / 1024**3, 2), "GB")

Dataset URL: https://www.kaggle.com/datasets/karthikgangula/credit-card-fraud-mega-dataset
extracted: credit_card_fraud.csv.zip
extracted: customers.csv.zip
credit_card_fraud.csv 10.42 GB
credit_card_fraud.csv.zip 2.09 GB
customers.csv 0.0 GB
customers.csv.zip 0.0 GB


In [0]:
"""Peek at the large file without a full scan / espia o arquivo grande sem varrer tudo."""

# read only the first rows to get the columns / lê só as primeiras linhas para ver as colunas
df_peek = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rf"{volume_path}/credit_card_fraud.csv")
    .limit(1000)
)
df_peek.printSchema()
display(df_peek.limit(10))

root
 |-- _c0: integer (nullable = true)
 |-- ssn: string (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- acct_num: long (nullable = true)
 |-- profile: string (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- trans_date: date (nullable = true)
 |-- trans_time: time(6) (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- merchant: string (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch

_c0,ssn,cc_num,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,acct_num,profile,trans_num,trans_date,trans_time,unix_time,category,amt,is_fraud,merchant,merch_lat,merch_long
0,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,52226c099d060def61a7e0d3de214e4b,2020-02-15,02:48:17,1581715097,misc_net,4.34,1,"fraud_Moore, Dibbert and Koepp",38.724545,-120.916415
1,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,6495d95c66d7d504d7352f2b8c105d3c,2020-02-15,01:29:36,1581710376,misc_net,11.2,1,fraud_Dach-Nader,37.288981,-121.791754
2,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,bc0bfcd9e0132b171e538192b0e27777,2020-02-15,11:40:04,1581747004,grocery_pos,282.55,1,fraud_Pacocha-O'Reilly,38.46153,-120.624583
3,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,483c446c7b56a11f0d4538056eb79984,2020-02-15,01:20:48,1581709848,gas_transport,294.94,1,"fraud_Kutch, Hermiston and Farrell",37.920539,-121.43939
4,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,fe2209222832f626cd2cfdcb6d7130b5,2020-02-15,01:15:19,1581709519,misc_net,326.4,1,fraud_Kuphal-Bartoletti,38.12385,-120.843755
5,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,d6a2e84b0338d887c0845c0ab943468c,2020-02-15,06:07:33,1581727053,grocery_pos,675.0,1,fraud_Auer-Mosciski,37.10285,-121.515057
6,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,f891e2d51f73adaf0055e029f331d703,2020-02-15,02:06:51,1581712611,grocery_pos,628.75,1,fraud_Strosin-Cruickshank,37.819908,-121.793001
7,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,eb63c1e4086b13d8daedefa089295acf,2020-02-15,03:49:05,1581718745,grocery_net,751.03,1,"fraud_Towne, Walker and Borer",37.034358,-121.629142
8,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,5d10fab6d2d2503e92f2abc1274e3aa1,2020-02-16,22:38:00,1581872880,shopping_net,994.33,1,fraud_Volkman-Predovic,37.958785,-121.960836
9,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,f5a701c2d709b135cd3d550bafe2e332,2020-02-16,23:09:54,1581874794,shopping_net,849.56,1,fraud_Kozey-Boehm,37.056368,-120.47054


In [0]:
"""Peek at the first rows of the large file / espia as primeiras linhas do arquivo grande."""

import pandas as pd

# volume path / caminho do volume
volume_path = r"/Volumes/workspace/bronze/raw_files"

# read only 1000 rows, no full scan / lê só 1000 linhas, sem varredura completa
df_peek = pd.read_csv(rf"{volume_path}/credit_card_fraud.csv", nrows=1000)

# columns, types and first rows / colunas, tipos e primeiras linhas
print(df_peek.dtypes)
display(df_peek.head(10))

Unnamed: 0      int64
ssn            object
cc_num          int64
first          object
last           object
gender         object
street         object
city           object
state          object
zip             int64
lat           float64
long          float64
city_pop        int64
job            object
dob            object
acct_num        int64
profile        object
trans_num      object
trans_date     object
trans_time     object
unix_time       int64
category       object
amt           float64
is_fraud        int64
merchant       object
merch_lat     float64
merch_long    float64
dtype: object


Unnamed: 0,ssn,cc_num,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,acct_num,profile,trans_num,trans_date,trans_time,unix_time,category,amt,is_fraud,merchant,merch_lat,merch_long
0,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,52226c099d060def61a7e0d3de214e4b,2020-02-15,02:48:17,1581715097,misc_net,4.34,1,"fraud_Moore, Dibbert and Koepp",38.724545,-120.916415
1,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,6495d95c66d7d504d7352f2b8c105d3c,2020-02-15,01:29:36,1581710376,misc_net,11.2,1,fraud_Dach-Nader,37.288981,-121.791754
2,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,bc0bfcd9e0132b171e538192b0e27777,2020-02-15,11:40:04,1581747004,grocery_pos,282.55,1,fraud_Pacocha-O'Reilly,38.46153,-120.624583
3,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,483c446c7b56a11f0d4538056eb79984,2020-02-15,01:20:48,1581709848,gas_transport,294.94,1,"fraud_Kutch, Hermiston and Farrell",37.920539,-121.43939
4,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,fe2209222832f626cd2cfdcb6d7130b5,2020-02-15,01:15:19,1581709519,misc_net,326.4,1,fraud_Kuphal-Bartoletti,38.12385,-120.843755
5,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,d6a2e84b0338d887c0845c0ab943468c,2020-02-15,06:07:33,1581727053,grocery_pos,675.0,1,fraud_Auer-Mosciski,37.10285,-121.515057
6,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,f891e2d51f73adaf0055e029f331d703,2020-02-15,02:06:51,1581712611,grocery_pos,628.75,1,fraud_Strosin-Cruickshank,37.819908,-121.793001
7,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,eb63c1e4086b13d8daedefa089295acf,2020-02-15,03:49:05,1581718745,grocery_net,751.03,1,"fraud_Towne, Walker and Borer",37.034358,-121.629142
8,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,5d10fab6d2d2503e92f2abc1274e3aa1,2020-02-16,22:38:00,1581872880,shopping_net,994.33,1,fraud_Volkman-Predovic,37.958785,-121.960836
9,750-09-7342,6011237648640631,Amanda,Morris,F,144 Bowers Route,Tracy,CA,95376,37.7383,-121.4345,106426,Teaching laboratory technician,2003-09-14,495733250797,young_adults_female_urban.json,f5a701c2d709b135cd3d550bafe2e332,2020-02-16,23:09:54,1581874794,shopping_net,849.56,1,fraud_Kozey-Boehm,37.056368,-120.47054


In [0]:
"""Inspect the customers file / inspeciona o arquivo de clientes."""

# read the small file with schema inference / lê o arquivo pequeno inferindo o schema
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(rf"{volume_path}/customers.csv")
)
df_customers.printSchema()
display(df_customers.limit(10))
print(df_customers.count())

root
 |-- ssn|cc_num|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|acct_num|profile: string (nullable = true)



ssn|cc_num|first|last|gender|street|city|state|zip|lat|long|city_pop|job|dob|acct_num|profile
115-04-4507|4218196001337|Jonathan|Johnson|M|863 Lawrence Valleys|Freeport|ME|04032|43.857|-70.1031|7879|Accounting technician|1973-10-03|888022315787|adults_50up_male_urban.json
715-55-5575|4351161559407816183|Edward|Fuller|M|310 Kendra Common Apt. 164|Chicago|IL|60638|41.7814|-87.7705|2680484|Professor Emeritus|1981-06-07|917558277935|adults_2550_male_urban.json
167-48-5821|4192832764832|Melinda|Cameron|F|05641 Robin Port|Reno|NV|89509|39.498|-119.8239|276896|International aid/development worker|1994-05-30|718172762479|adults_2550_female_urban.json
406-83-7518|4238849696532874|Brandon|Williams|M|26916 Carlson Mountain|Lady Lake|FL|32162|28.9471|-81.9971|74047|Seismic interpreter|1971-12-26|947268892251|adults_50up_male_urban.json
697-93-1877|4514627048281480|Lee|Hernandez|M|809 Burns Creek|Hermosa Beach|CA|90254|33.8643|-118.3955|19506|Medical laboratory scientific officer|1966-05-22|888335239225|adults_50up_male_urban.json
246-08-3947|3517182278248964|Tracey|Hickman|F|465 Lam Mission|Los Angeles|CA|90011|34.0079|-118.2582|2383912|Community arts worker|1972-06-18|245877882397|adults_50up_female_urban.json
347-03-9639|213193010310510|Jose|Ramirez|M|82997 Kristen Valley Suite 311|Paris|TX|75462|33.6805|-95.4905|34942|Plant breeder/geneticist|1984-03-06|465378164447|adults_2550_male_urban.json
479-94-0888|4065133387262473|David|Leblanc|M|108 Tanya Terrace Apt. 267|Lancaster|PA|17601|40.0766|-76.3107|164596|Quarry manager|1981-12-27|950105810049|adults_2550_male_urban.json
061-22-6210|630447468723|Colton|Robinson|M|805 Brendan Neck|Lake|MI|48632|43.8575|-85.0219|5125|Retail buyer|2001-08-23|940251608167|young_adults_male_urban.json
162-08-8321|2222913619399092|Angela|Sandoval|F|3534 Anderson Rue Suite 751|Brownstown|IL|62418|38.9891|-88.9494|2133|Acupuncturist|1971-01-26|684868121383|adults_50up_female_rural.json


20000


In [0]:
"""Remove the zip files after extraction / remove os zips depois da extração."""

# delete only the compressed copies / apaga somente as cópias compactadas
for f in dbutils.fs.ls(volume_path):
    if f.name.endswith(".zip"):
        dbutils.fs.rm(rf"{volume_path}/{f.name}")
        print("removed:", f.name)

removed: credit_card_fraud.csv.zip
removed: customers.csv.zip
